In [5]:
import tensorflow as tf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


In [6]:
df = pd.read_csv("data/ECG_10s_2500.csv")

In [7]:

hypo_patients = [1, 2, 4, 7, 8]  
normal_only_patients = [3, 5, 6, 9]  


def normalize_segments(X):

    mean = np.mean(X, axis=1, keepdims=True)
    std = np.std(X, axis=1, keepdims=True) + 1e-8
    return (X - mean) / std

results = []

for test_patient in hypo_patients:
    print(f"\n=== Test Patient: {test_patient} ===")

    train_patients = [p for p in hypo_patients + normal_only_patients if p != test_patient]
    
    
    train_df = df[df['patient_id'].isin(train_patients)]
    test_df = df[df['patient_id'] == test_patient]
    
    val_patient = np.random.choice([p for p in hypo_patients if p not in [test_patient]])
    val_df = train_df[train_df['patient_id'] == val_patient]
    train_df = train_df[train_df['patient_id'] != val_patient]

    print(f"Train patients: {sorted(train_df['patient_id'].unique())}")
    print(f"Validation patient: {val_patient}")
    print(f"Test patient: {test_patient}")

  
    X_train = train_df.iloc[:, 0:2500].values
    y_train = train_df['label'].values

    X_val = val_df.iloc[:, 0:2500].values
    y_val = val_df['label'].values

    X_test = test_df.iloc[:, 0:2500].values
    y_test = test_df['label'].values

  
    X_train = normalize_segments(X_train)
    X_val = normalize_segments(X_val)
    X_test = normalize_segments(X_test)

   
    X_train = X_train.reshape(-1, 2500, 1)
    X_val = X_val.reshape(-1, 2500, 1)
    X_test = X_test.reshape(-1, 2500, 1)

    print(f"Train shape: {X_train.shape}, Val shape: {X_val.shape}, Test shape: {X_test.shape}")

    
    from sklearn.utils.class_weight import compute_class_weight

    class_weights_values = compute_class_weight(
        class_weight='balanced',
        classes=np.unique(y_train),
        y=y_train
    )
    class_weights = {i: w for i, w in enumerate(class_weights_values)}

    print("Class weights:", class_weights)

    



=== Test Patient: 1 ===
Train patients: [np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(8), np.int64(9)]
Validation patient: 7
Test patient: 1
Train shape: (30698, 2500, 1), Val shape: (14363, 2500, 1), Test shape: (569, 2500, 1)
Class weights: {0: np.float64(0.5424825051247615), 1: np.float64(6.38477537437604)}

=== Test Patient: 2 ===
Train patients: [np.int64(1), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(8), np.int64(9)]
Validation patient: 7
Test patient: 2
Train shape: (26869, 2500, 1), Val shape: (14363, 2500, 1), Test shape: (4398, 2500, 1)
Class weights: {0: np.float64(0.5420853004075374), 1: np.float64(6.440316395014381)}

=== Test Patient: 4 ===
Train patients: [np.int64(1), np.int64(2), np.int64(3), np.int64(5), np.int64(6), np.int64(7), np.int64(9)]
Validation patient: 8
Test patient: 4
Train shape: (26744, 2500, 1), Val shape: (16560, 2500, 1), Test shape: (2326, 2500, 1)
Class weights: {0: np.float64(0.5081512445373361), 1: n

In [8]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, BatchNormalization, Dropout
from tensorflow.keras.layers import Bidirectional, LSTM, Dense
from tensorflow.keras.optimizers import Adam
import tensorflow as tf

def build_cnn_bilstm(input_shape=(2500, 1)):
    model = Sequential([
        # --- CNN Feature Extraction ---
        Conv1D(filters=64, kernel_size=7, activation='relu', padding='same', input_shape=input_shape),
        BatchNormalization(),
        MaxPooling1D(pool_size=2),
        Dropout(0.3),

        Conv1D(filters=128, kernel_size=5, activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling1D(pool_size=2),
        Dropout(0.3),

        # --- BiLSTM Temporal Modeling ---
        Bidirectional(LSTM(64, return_sequences=True)),
        Dropout(0.3),
        Bidirectional(LSTM(32)),

        # --- Dense Layers ---
        Dense(64, activation='relu'),
        Dropout(0.3),
        Dense(1, activation='sigmoid')  # Binary classification
    ])

    model.compile(
        optimizer=Adam(learning_rate=1e-3),
        loss='binary_crossentropy',
        metrics=['accuracy', tf.keras.metrics.AUC(name='auc'), tf.keras.metrics.Recall(name='recall')]
    )

    return model

In [12]:
from tensorflow.keras.callbacks import EarlyStopping

# Early stopping to prevent overfitting
early_stop = EarlyStopping(monitor='val_auc', patience=5, restore_best_weights=True, mode='max')

# Build a fresh model for this fold
model = build_cnn_bilstm(input_shape=(2500, 1))
print(model.summary())

# Train model
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=30,
    batch_size=64,
    class_weight=class_weights,
    callbacks=[early_stop],
    verbose=1
)

# Evaluate on the held-out test patient
test_loss, test_acc, test_auc, test_recall = model.evaluate(X_test, y_test, verbose=0)
print(f"Test Accuracy: {test_acc:.3f} | AUC: {test_auc:.3f} | Recall: {test_recall:.3f}")

# Save results
results.append({
    'test_patient': test_patient,
    'accuracy': test_acc,
    'auc': test_auc,
    'recall': test_recall
})

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv1d_6 (Conv1D)                    │ (None, 2500, 64)            │             512 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_6                │ (None, 2500, 64)            │             256 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling1d_6 (MaxPooling1D)       │ (None, 1250, 64)            │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_12 (Dropout)                 │ (None, 1250, 64)            │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv1d_7 (Conv1D)                    │ (None, 1250, 128)           │          41,088 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_7                │ (None, 1250, 128)           │             512 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling1d_7 (MaxPooling1D)       │ (None, 625, 128)            │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_13 (Dropout)                 │ (None, 625, 128)            │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ bidirectional_6 (Bidirectional)      │ (None, 625, 128)            │          98,816 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_14 (Dropout)                 │ (None, 625, 128)            │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ bidirectional_7 (Bidirectional)      │ (None, 64)                  │          41,216 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_6 (Dense)                      │ (None, 64)                  │           4,160 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_15 (Dropout)                 │ (None, 64)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_7 (Dense)                      │ (None, 1)                   │              65 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 186,625 (729.00 KB)

 Trainable params: 186,241 (727.50 KB)

 Non-trainable params: 384 (1.50 KB)

None
Epoch 1/30
446/446 ━━━━━━━━━━━━━━━━━━━━ 726s 2s/step - accuracy: 0.6591 - auc: 0.7526 - loss: 0.5851 - recall: 0.7203 - val_accuracy: 0.4359 - val_auc: 0.3390 - val_loss: 0.8904 - val_recall: 0.3148
Epoch 2/30
446/446 ━━━━━━━━━━━━━━━━━━━━ 703s 2s/step - accuracy: 0.7329 - auc: 0.8446 - loss: 0.4810 - recall: 0.8172 - val_accuracy: 0.3076 - val_auc: 0.2171 - val_loss: 1.1650 - val_recall: 0.2593
Epoch 3/30
446/446 ━━━━━━━━━━━━━━━━━━━━ 682s 2s/step - accuracy: 0.7006 - auc: 0.8471 - loss: 0.4607 - recall: 0.8527 - val_accuracy: 0.3462 - val_auc: 0.2606 - val_loss: 0.9542 - val_recall: 0.2407
Epoch 4/30
446/446 ━━━━━━━━━━━━━━━━━━━━ 693s 2s/step - accuracy: 0.7682 - auc: 0.8886 - loss: 0.3953 - recall: 0.9045 - val_accuracy: 0.1072 - val_auc: 0.6328 - val_loss: 1.7270 - val_recall: 0.9815
Epoch 5/30
446/446 ━━━━━━━━━━━━━━━━━━━━ 695s 2s/step - accuracy: 0.7918 - auc: 0.9102 - loss: 0.3455 - recall: 0.9318 - val_accuracy: 0.6837 - val_auc: 0.2325 - val_loss: 0.6715 - val_recall: 0.0556


In [13]:
tf.random.set_seed(42)
np.random.seed(42)